# Early-Exit CNN → Quantised TFLite → C (Apache TVM)

**Author:** Ajay Kumar M, Vishnu PS

**Pipeline**
1. Build / load a backbone from the model registry
2. Fine-tune on CIFAR-10
3. Select the exit layers with Algorithm 1 of the paper (hardware-aware exit placement; optionally accuracy-aware
   or with measured layer costs; `software/dynn_placement.py` of the repository)
4. Attach and jointly train exit heads (backbone frozen)
5. Quantise to INT8 TFLite and compare with FP32
6. Compile to C source via Apache TVM

## 0 · Dependencies (run once, then RESTART RUNTIME)

In [ ]:
!pip install -q tensorflow==2.12.0 keras==2.12.0 tflite==2.1.0 numpy==1.23.5
!pip install -q apache-tvm==0.14.dev273
# RESTART THE RUNTIME HERE. Dependency-resolver errors during install are expected
# and can be ignored provided the version check in the next cell passes.

In [ ]:
!pip install "https://files.pythonhosted.org/packages/9c/b2/adb56267ba6b6f05204407a7745e0c188e39589b7e33f490d4aa406d84e8/apache_tvm-0.14.dev273-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl"

## 1 · Configuration

In [ ]:
import os, pathlib, itertools
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tvm, tvm.relay as relay
from tvm.relay.backend import Executor, Runtime
from tvm.micro import export_model_library_format

SEED = 42
tf.random.set_seed(SEED); np.random.seed(SEED)

# ---- Paths -------------------------------------------------------------
DRIVE_ROOT   = "/content/drive/MyDrive/"
CACHE_DIR    = f"{DRIVE_ROOT}/cached_datasets"
WEIGHTS_DIR  = f"{DRIVE_ROOT}/finetuned_weights"
CONVERTED_DIR= f"{DRIVE_ROOT}/converted_models"

# ---- Data --------------------------------------------------------------
IMAGE_SIZE    = (64, 64, 3)
BATCH_SIZE    = 128
NUM_CLASSES   = 10
CIFAR_CLASSES = ['airplane','automobile','bird','cat','deer',
                 'dog','frog','horse','ship','truck']

# ---- Backbone ----------------------------------------------------------
STEM_STRIDE   = 2

# ---- Training ----------------------------------------------------------
BASE_OPTIMIZER  = "sgd"
BASE_LR         = 0.1
LR_SCHEDULE     = "cosine"
LR_DECAY_AT     = (0.5, 0.75)
LR_DECAY_FACTOR = 0.1
EPOCHS_FINETUNE = 100
AUGMENT         = True

LEARNING_RATE   = 5e-4
EPOCHS_EE       = 10
DROPOUT_RATE    = 0.1
FINETUNE_FROM_SCRATCH = False      # False -> load weights from WEIGHTS_DIR

FREEZE_FINAL_HEAD = True

# ---- Early exit --------------------------------------------------------
EE_LOSS_WEIGHTS   = [0.2, 0.4, 0.7, 1.0]
EXIT_HEAD_STRIDE  = 1
# Exit placement: Algorithm 1 of the paper (software/dynn_placement.py)
AUTO_SELECT_EXITS = True        # False = use exit_layers from the registry
PLACEMENT_MODE    = "depth"     # "depth":    depth-based selection (the networks of the paper)
                                # "accuracy": accuracy-aware option (Algorithm 1, line 17): an exit head is trained at
                                #             every candidate layer and evaluated on the validation split
ALPHA_ACC         = 0.25        # accelerator cost factor on the BB core (1.0 = no accelerator)
MEASURED_COSTS    = None        # or the measured cost of every backbone layer on the target (Algorithm 1, line 1),
                                # e.g. "software/measured_costs/ResNet8_DEA.json"; replaces ALPHA_ACC
N_EXITS_TARGET    = 3           # N_max; the search reduces N if no configuration is feasible
N_EXITS_MIN       = 1
MIN_FIRST_DEPTH   = 10.0        # D_min (% of the backbone cost)
LAST_DEPTH_RANGE  = (55.0, 90.0)  # [D_L,min, D_L,max]
TARGET_LAST_DEPTH = 70.0        # D_tar
LAMBDA_BAL        = 10.0        # lambda, weight of the segment-balance term
ACC_MARGIN        = 2.0         # Delta of the accuracy-aware option (accuracy points below the no-exit accuracy)
# Candidate exits: layers with a 4-D output; only the architectural boundaries below if there are >= 2 * N_max
BOUNDARY_KEYWORDS       = ["add", "_out", "_pool", "_concat", "pool_conv", "project/batchnorm"]
BOUNDARY_CASE_SENSITIVE = False
BOUNDARY_LAYER_TYPES    = (tf.keras.layers.Concatenate, tf.keras.layers.MaxPooling2D,
                           tf.keras.layers.AveragePooling2D)
BOUNDARY_DEDUP          = True  # of layers with equal cumulative cost (Activation after Add), keep the last
REPO_REF          = "main"      # branch or tag of the repository from which dynn_placement.py is loaded

# ---- TVM ---------------------------------------------------------------
TVM_OPT_LEVEL    = 3
QUANTIZE_FOR_TVM = True

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
for d in (CACHE_DIR, WEIGHTS_DIR, CONVERTED_DIR):
    os.makedirs(d, exist_ok=True)

In [ ]:
# Algorithm 1 (dynn_placement.py) and the measured layer costs are in the folder software/ of the repository.
import sys, json, hashlib, urllib.request
REPO_URL = f"https://raw.githubusercontent.com/ajaymahendersingh/DyNN_RV_SoC/{REPO_REF}/"

def repo_file(relpath):
    """Path of a file of the repository: the local copy if the notebook runs in a clone, else downloaded once."""
    for p in (pathlib.Path(relpath), pathlib.Path(relpath).relative_to("software") if relpath.startswith("software/")
              else None, pathlib.Path(pathlib.Path(relpath).name)):
        if p is not None and p.exists():
            return str(p)
    urllib.request.urlretrieve(REPO_URL + relpath, pathlib.Path(relpath).name)
    return pathlib.Path(relpath).name

sys.path.insert(0, os.path.dirname(os.path.abspath(repo_file("software/dynn_placement.py"))))
import dynn_placement as DP

## 2 · Backbone definitions

In [ ]:
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, Activation,
                                     Add, Dense)
from tensorflow.keras.regularizers import l2

def _conv_bn(x, filters, k, strides, name, bn=True):
    x = Conv2D(filters, k, strides=strides, padding="same",
               kernel_initializer="he_normal", kernel_regularizer=l2(1e-4),
               name=name)(x)
    if bn:
        x = BatchNormalization(name=name + "_bn")(x)
    return x

def resnet_v1_cifar(input_shape=IMAGE_SIZE, n=1, num_filters=16, stem_stride=STEM_STRIDE):
    '''MLPerf Tiny ResNet-8 (v1) feature extractor, include_top=False.'''
    inp = Input(shape=input_shape, name="input")
    # 64x64 input -> 32x32 feature map
    x = _conv_bn(inp, num_filters, 3, stem_stride, "stem_conv")
    x = Activation("relu", name="stem_out")(x)

    for s in range(1, 4):
        f = num_filters * 2 ** (s - 1)
        for b in range(1, n + 1):
            stride = 2 if (s > 1 and b == 1) else 1
            tag = f"stack{s}_block{b}"
            y = _conv_bn(x, f, 3, stride, f"{tag}_conv1")
            y = Activation("relu", name=f"{tag}_relu1")(y)
            y = _conv_bn(y, f, 3, 1, f"{tag}_conv2")
            if stride != 1 or x.shape[-1] != f:
                x = _conv_bn(x, f, 1, stride, f"{tag}_proj", bn=False)
            x = Add(name=f"{tag}_add")([x, y])
            x = Activation("relu", name=f"{tag}_out")(x)
    return tf.keras.Model(inp, x, name=f"resnet{6 * n + 2}_v1")

## 3 · Model registry

In [ ]:
def _load_h5(path):
    return lambda: tf.keras.models.load_model(path)

MODEL_REGISTRY = {
    "resnet8": {
        "name": "ResNet8",
        "build_fn": lambda: resnet_v1_cifar(IMAGE_SIZE, 1, 16, STEM_STRIDE),
        "exit_layers": ["stack1_block1_out", "stack2_block1_out"],
        "head_offset": -1,
        "max_exits": 2,
        "tag": f"stem{STEM_STRIDE}",
    },

    "mobilenet_v1": {
        "name": "MobileNetV1",
        "build_fn": lambda: tf.keras.applications.MobileNet(
            input_shape=IMAGE_SIZE, alpha=0.25, weights="imagenet", include_top=False),
        "exit_layers": ["conv_pw_2_relu", "conv_pw_4_relu", "conv_pw_6_relu"],
        "head_offset": -5,
    },
    "mobilenet_v2": {
        "name": "MobileNetV2",
        "build_fn": lambda: tf.keras.applications.MobileNetV2(
            input_shape=IMAGE_SIZE, alpha=0.35, weights="imagenet", include_top=False),
        "exit_layers": ["block_3_expand_relu", "block_6_expand_relu", "block_10_expand_relu"],
        "head_offset": -1,
    },
    "efficientnet_b0": {
        "name": "EfficientNetB0",
        "build_fn": lambda: tf.keras.applications.EfficientNetB0(
            input_shape=IMAGE_SIZE, weights="imagenet", include_top=False),
        "exit_layers": ["block2a_expand_activation", "block3a_expand_activation",
                        "block5a_expand_activation"],
        "head_offset": -1,
        "expects_0_255": True,
    },
    "resnet50": {
        "name": "ResNet50",
        "build_fn": lambda: tf.keras.applications.ResNet50(
            input_shape=IMAGE_SIZE, weights="imagenet", include_top=False),
        "exit_layers": ["conv2_block3_out", "conv3_block4_out", "conv4_block6_out"],
        "head_offset": -1,
    },
    "vgg16": {
        "name": "VGG16",
        "build_fn": lambda: tf.keras.applications.VGG16(
            input_shape=IMAGE_SIZE, weights="imagenet", include_top=False),
        "exit_layers": ["block2_pool", "block3_pool", "block4_pool"],
        "head_offset": -1,
    },
    "nasnet_mobile": {
        "name": "NASNetMobile",
        "build_fn": lambda: tf.keras.applications.NASNetMobile(
            input_shape=IMAGE_SIZE, weights="imagenet", include_top=False),
        "exit_layers": ["normal_concat_3", "normal_concat_6", "normal_concat_9"],
        "head_offset": -1,
        "tvm_custom_pipeline": False,   # branchy graph hangs the fold-before-QNN pipeline
        "tvm_opt_level": 1,
    },
}

In [ ]:
# ---------------- CHOOSE MODEL ----------------
MODEL_KEY = "resnet8"
# ----------------------------------------------
MODEL_CFG  = MODEL_REGISTRY[MODEL_KEY]
# One run directory per model and placement setting, so that runs with different exits do not overwrite each other.
PLACEMENT_TAG = ("registry" if not AUTO_SELECT_EXITS else
                 ("measured" if MEASURED_COSTS else f"alpha{ALPHA_ACC:g}")
                 + ("_accuracy" if PLACEMENT_MODE == "accuracy" else ""))
OUTPUT_DIR = pathlib.Path(f"./output_{MODEL_CFG['name']}_{PLACEMENT_TAG}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_EXITS_TARGET = min(N_EXITS_TARGET, MODEL_CFG.get("max_exits", N_EXITS_TARGET))
print(f"Model: {MODEL_CFG['name']} | exits<= {N_EXITS_TARGET} | out: {OUTPUT_DIR}")

## 4 · Data

In [ ]:
cache_path = os.path.join(CACHE_DIR, f"cifar{NUM_CLASSES}_{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}.npz")

if os.path.exists(cache_path):
    d = np.load(cache_path)
    x_train_all, y_train_all = d["x_train"], d["y_train"]
    x_test_all,  y_test_all  = d["x_test"],  d["y_test"]
    print(f"Loaded cache {cache_path}")
else:
    (xtr, ytr), (xte, yte) = tf.keras.datasets.cifar10.load_data()
    x_train_all = tf.image.resize(xtr.astype("float32") / 255.0, IMAGE_SIZE[:2]).numpy()
    x_test_all  = tf.image.resize(xte.astype("float32") / 255.0, IMAGE_SIZE[:2]).numpy()
    y_train_all = tf.keras.utils.to_categorical(ytr.squeeze(), NUM_CLASSES)
    y_test_all  = tf.keras.utils.to_categorical(yte.squeeze(), NUM_CLASSES)
    np.savez_compressed(cache_path, x_train=x_train_all, y_train=y_train_all,
                        x_test=x_test_all, y_test=y_test_all)
    print(f"Cached -> {cache_path}")

idx = np.random.default_rng(SEED).permutation(len(x_train_all))
n_tr = int(0.8 * len(idx))
tr_idx, va_idx = idx[:n_tr], idx[n_tr:]

AUGMENT_FN = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal", seed=SEED),
    tf.keras.layers.RandomTranslation(0.125, 0.125, fill_mode="reflect", seed=SEED),
], name="augment")

def _ds(x, y, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y)).batch(BATCH_SIZE)
    if augment:
        ds = ds.map(lambda a, b: (AUGMENT_FN(a, training=True), b),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = _ds(x_train_all[tr_idx], y_train_all[tr_idx], augment=AUGMENT)
val_ds   = _ds(x_train_all[va_idx], y_train_all[va_idx])
full_ds  = _ds(x_train_all, y_train_all)
x_val, y_val = x_train_all[va_idx], y_train_all[va_idx]     # validation split (accuracy-aware placement)
test_images, test_labels = x_test_all, y_test_all

print(f"train {len(train_ds)} / val {len(val_ds)} / full {len(full_ds)} batches, "
      f"test {len(test_images)} images")

## 5 · Fine-tune the backbone

In [ ]:
pretrained = MODEL_CFG["build_fn"]()
backbone_model = tf.keras.Model(
    inputs=pretrained.inputs,
    outputs=pretrained.layers[MODEL_CFG["head_offset"]].output,
    name=f"{MODEL_CFG['name']}_backbone",
)
assert len(backbone_model.output_shape) == 4, (
    f"head_offset={MODEL_CFG['head_offset']} does not yield a 4-D feature map "
    f"(got {backbone_model.output_shape}); fix the registry entry.")

layers = [tf.keras.layers.InputLayer(input_shape=IMAGE_SIZE)]
if MODEL_CFG.get("expects_0_255"):
    layers.append(tf.keras.layers.Rescaling(255.0))
layers += [
    backbone_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(DROPOUT_RATE),
    tf.keras.layers.Dense(NUM_CLASSES, activation="softmax"),
]
base_model = tf.keras.Sequential(layers, name=f"{MODEL_CFG['name']}_base")
base_model.compile(
    optimizer=(tf.keras.optimizers.SGD(BASE_LR, momentum=0.9, nesterov=True)
               if BASE_OPTIMIZER == "sgd" else tf.keras.optimizers.Adam(BASE_LR)),
    loss="categorical_crossentropy", metrics=["accuracy"])
base_model.summary()

In [ ]:
RUN_TAG = f"{MODEL_CFG['name']}_cifar{NUM_CLASSES}_{IMAGE_SIZE[0]}" \
          + (f"_{MODEL_CFG['tag']}" if MODEL_CFG.get("tag") else "")
weights_path = os.path.join(WEIGHTS_DIR, f"{RUN_TAG}.weights.h5")

def lr_at(epoch):
    if LR_SCHEDULE == "cosine":
        return float(BASE_LR * 0.5 * (1 + np.cos(np.pi * epoch / EPOCHS_FINETUNE)))
    lr = BASE_LR
    for frac in LR_DECAY_AT:
        if epoch >= int(frac * EPOCHS_FINETUNE):
            lr *= LR_DECAY_FACTOR
    return lr

print(f"LR '{LR_SCHEDULE}' over {EPOCHS_FINETUNE} epochs: "
      f"start {lr_at(0):.4g} | mid {lr_at(EPOCHS_FINETUNE // 2):.4g} | "
      f"end {lr_at(EPOCHS_FINETUNE - 1):.4g}")
assert lr_at(EPOCHS_FINETUNE - 1) < 0.05 * BASE_LR, \
    "LR never decays over this budget. Check LR_SCHEDULE / EPOCHS_FINETUNE."

if FINETUNE_FROM_SCRATCH:
    history = base_model.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS_FINETUNE, verbose=2,
        callbacks=[tf.keras.callbacks.LearningRateScheduler(lr_at),
                   tf.keras.callbacks.ModelCheckpoint(
                       weights_path, monitor="val_accuracy", mode="max",
                       save_best_only=True, save_weights_only=True)])
    base_model.load_weights(weights_path)   # best epoch, not the last one
    print(f"Best weights -> {weights_path}")
else:
    base_model.load_weights(weights_path)
    history = None
    print(f"Loaded weights <- {weights_path}")

base_eval = base_model.evaluate(_ds(test_images, test_labels), verbose=0)
print(f"Single-exit test accuracy: {base_eval[1]:.4f}")

In [ ]:
def plot_history(hist, title="", save_path=None):
    if hist is None:
        print(f"No history for '{title}' - skipped.")
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, m in zip(axes, ["loss", "accuracy"]):
        train_key, val_key = m, f"val_{m}"
        if train_key not in hist.history:
            cands = [k for k in hist.history
                     if k.endswith(f"_{m}") and not k.startswith("val_")]
            if not cands:
                continue
            train_key, val_key = cands[-1], f"val_{cands[-1]}"
        ax.plot(hist.history[train_key], label="train")
        ax.plot(hist.history[val_key], label="val")
        ax.set(xlabel="Epoch", ylabel=m.title(), title=f"{title}{m.title()}")
        ax.legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

plot_history(history, f"{MODEL_CFG['name']} base - ",
             OUTPUT_DIR / f"{MODEL_CFG['name']}_base_history.png")

In [ ]:
backbone = next(l for l in base_model.layers if isinstance(l, tf.keras.Model))
backbone.summary()

## 6 · Exit head

In [ ]:
def make_exit_head(features, name, exit_idx=1):
    '''Exit head. Deeper exits get one extra 3x3 conv.'''
    x = tf.keras.layers.Conv2D(32, 3, strides=EXIT_HEAD_STRIDE, padding="same",
                               activation="relu", name=f"conv1_{name}")(features)
    if exit_idx >= 2:
        x = tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu",
                                   name=f"conv2_{name}")(x)
    x = tf.keras.layers.GlobalAveragePooling2D(name=f"gap_{name}")(x)
    x = tf.keras.layers.Dense(128, activation="relu", name=f"dense_mid_1_{name}")(x)
    x = tf.keras.layers.Dense(64, activation="relu", name=f"dense_mid_2_{name}")(x)
    return tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name=name)(x)

## 7 · Exit placement (Algorithm 1)

**Cost model.** Only the BB core reaches the dot-product accelerator. Backbone MACs that are accelerator-eligible
(1x1 Conv2D, Dense) are scaled by `ALPHA_ACC`; every other backbone MAC and *all* exit-head MACs are charged at full
software cost, because the EE core runs the heads without the accelerator. If the network has been profiled on the
target, `MEASURED_COSTS` replaces this model by the measured cost of every layer (line 1 of Algorithm 1; the head
cost is then the measured cycles per MAC of the EE core times the head MACs). Measured costs of the five networks of
the paper on DE+A are in `software/measured_costs/`.

**Depth-based selection.** The first exit must lie past `MIN_FIRST_DEPTH` % of the backbone cost and the last exit in
`LAST_DEPTH_RANGE`; every exit head must cost less than the backbone segment that follows it (for the last exit, the
rest of the backbone), so that the EE core finishes before the BB core reaches the next exit. Among the feasible
configurations, the one whose last exit is closest to `TARGET_LAST_DEPTH` and whose segments are best balanced
(`LAMBDA_BAL`) is selected. If no configuration with `N_EXITS_TARGET` exits is feasible, N is reduced down to
`N_EXITS_MIN`. With the default settings, this gives the exits of the networks in the paper.

**Accuracy-aware option** (`PLACEMENT_MODE = "accuracy"`, line 17 of Algorithm 1): see the next cell.

In [ ]:
if AUTO_SELECT_EXITS:
    # One row per backbone layer: name, output shape, accelerator-eligible and software MACs, boundary flag
    table = DP.layer_table(backbone, BOUNDARY_KEYWORDS, BOUNDARY_CASE_SENSITIVE, BOUNDARY_LAYER_TYPES)
    if MEASURED_COSTS:
        layer_cost, head_per_mac = DP.load_measured_costs(repo_file(MEASURED_COSTS), table)
        head_cost = lambda hwc, k: head_per_mac * DP.probe_head_cost(make_exit_head, hwc, k)
        print(f"Measured layer costs: {MEASURED_COSTS} ({head_per_mac:.2f} cycles per head MAC)")
    else:
        layer_cost = None
        head_cost = lambda hwc, k: DP.probe_head_cost(make_exit_head, hwc, k)   # MACs of make_exit_head
    PLACEMENT_N, PLACEMENT_CONFIGS, cand, cum = DP.place(
        table, head_cost, alpha=ALPHA_ACC, layer_cost=layer_cost, d_min=MIN_FIRST_DEPTH,
        last_range=LAST_DEPTH_RANGE, d_tar=TARGET_LAST_DEPTH, lam=LAMBDA_BAL,
        n_max=N_EXITS_TARGET, n_min=N_EXITS_MIN, dedup=BOUNDARY_DEDUP)
    print(f"Backbone: {len(table['name'])} layers, total cost {cum[-1]:,.0f}"
          + ("" if MEASURED_COSTS else f" (alpha_acc={ALPHA_ACC})") + f"; {len(cand)} candidate exits (* boundary)")
    for i in cand:
        print(f"  [{i:3d}] {table['name'][i]:<45s} shape={table['hwc'][i]}  "
              f"depth={100 * cum[i] / cum[-1]:5.1f}%{' *' if table['boundary'][i] else ''}")
    if PLACEMENT_N is None:
        print(f"\nNo feasible placement for N in [{N_EXITS_MIN}, {N_EXITS_TARGET}]: "
              f"using exit_layers from the registry: {MODEL_CFG['exit_layers']}")
    else:
        print(f"\nN = {PLACEMENT_N}: {len(PLACEMENT_CONFIGS)} feasible configuration(s), best by (phi, psi):")
        DP.print_options(PLACEMENT_CONFIGS)
        MODEL_CFG["exit_layers"] = PLACEMENT_CONFIGS[0]["exits"]
        print(f"\nDepth-based selection: {MODEL_CFG['exit_layers']}")
else:
    PLACEMENT_N, PLACEMENT_CONFIGS = None, []
    print(f"AUTO_SELECT_EXITS = False: using exit_layers from the registry: {MODEL_CFG['exit_layers']}")
MODEL_CFG["n_exits_used"] = len(MODEL_CFG["exit_layers"])

### Accuracy-aware option

Runs only with `PLACEMENT_MODE = "accuracy"`. An exit head is trained at every layer that occurs in a feasible
configuration (the exit-1 head where the layer is a first exit, the deeper head where it is a later exit), with the
backbone frozen and the optimizer, learning rate and epochs of the exit training below. Because the backbone is
frozen, the heads are independent, so they are trained together in one model. All heads are then quantized to INT8
together with the network, as deployed, and early exiting is simulated on the validation split for the nine threshold
profiles of the paper: an image leaves at the first exit whose confidence reaches the threshold and costs the backbone
up to that exit plus the head; otherwise it costs the whole backbone. The configuration with the lowest mean cost
per image at an accuracy `ACC_MARGIN` points below the accuracy without exits (interpolated between the profiles) is
selected. This takes longer than the depth-based selection (one training of all candidate heads).

In [ ]:
if AUTO_SELECT_EXITS and PLACEMENT_MODE == "accuracy" and PLACEMENT_N is not None:
    pairs = DP.head_pairs(PLACEMENT_CONFIGS)                     # (layer, head type) to train
    taps_at = sorted({n for n, _ in pairs}, key=table["name"].index)
    print(f"{len(PLACEMENT_CONFIGS)} configurations: {len(pairs)} exit heads at {len(taps_at)} layers")

    # One model: frozen backbone, final classifier of the fine-tuned model, one head per (layer, head type)
    backbone.trainable = False
    tail = base_model.layers[base_model.layers.index(backbone) + 1:]
    tail_trainable = [l.trainable for l in tail]
    for l in tail:
        l.trainable = False
    inp = tf.keras.Input(shape=IMAGE_SIZE, name="image")
    x = tf.keras.layers.Rescaling(255.0, name="rescale_to_255")(inp) if MODEL_CFG.get("expects_0_255") else inp
    taps = tf.keras.Model(backbone.input, [backbone.get_layer(n).output for n in taps_at] + [backbone.output],
                          name="candidate_taps")(x)
    head_name = {p: f"cand_{taps_at.index(p[0])}_{p[1]}" for p in pairs}
    heads = [make_exit_head(taps[taps_at.index(p[0])], head_name[p], exit_idx=p[1]) for p in pairs]
    final = taps[-1]
    for l in tail:
        final = l(final)
    cand_model = tf.keras.Model(inp, [final] + heads, name="candidate_heads")
    final_name = cand_model.output_names[0]
    cand_model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
                       loss={n: (None if n == final_name else "categorical_crossentropy")
                             for n in cand_model.output_names})
    cand_model.fit(train_ds, epochs=EPOCHS_EE, verbose=2)
    for l, t in zip(tail, tail_trainable):
        l.trainable = t

    # INT8 as deployed (converter settings of the quantisation step below); outputs matched by name
    def calibration_images():
        for batch, _ in full_ds.take(100):
            yield [batch]
    conv = tf.lite.TFLiteConverter.from_keras_model(cand_model)
    conv.optimizations             = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset    = calibration_images
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type      = tf.float32
    conv.inference_output_type     = tf.float32
    out = DP.run_tflite(conv.convert(), x_val)
    y_v = np.argmax(y_val, axis=1)
    head_outputs = {p: (out[head_name[p]].max(1), out[head_name[p]].argmax(1)) for p in pairs}

    sel = DP.select_accuracy_aware(PLACEMENT_CONFIGS, cum, cum[-1], head_outputs, out[final_name].argmax(1), y_v,
                                   ACC_MARGIN)
    print(f"\nValidation split ({len(y_v)} images), INT8; C_Delta in % of the backbone cost:")
    for c in sorted(PLACEMENT_CONFIGS, key=lambda c: c["c_delta"])[:5] + [PLACEMENT_CONFIGS[0]]:
        acc = [float((head_outputs[(n, 1 if k == 0 else 2)][1] == y_v).mean() * 100) for k, n in enumerate(c["exits"])]
        tag = "  <- depth-based" if c is PLACEMENT_CONFIGS[0] else ("  <- selected" if c is sel else "")
        print(f"  C_Delta {100 * c['c_delta'] / cum[-1]:5.1f}%  {c['exits']}  head acc {[round(a, 1) for a in acc]}  "
              f"best-EE {c['best']['profile']} ({c['best']['acc']:.1f}%){tag}")
    MODEL_CFG["exit_layers"] = sel["exits"]
    MODEL_CFG["n_exits_used"] = len(sel["exits"])
    print(f"\nAccuracy-aware selection: {sel['exits']}")

In [ ]:
# Record of the exit selection, next to the outputs of this run
if AUTO_SELECT_EXITS and PLACEMENT_N is not None:
    rec = dict(model=MODEL_CFG["name"], mode=PLACEMENT_MODE, alpha_acc=ALPHA_ACC, measured_costs=MEASURED_COSTS,
               d_min=MIN_FIRST_DEPTH, last_range=LAST_DEPTH_RANGE, d_tar=TARGET_LAST_DEPTH, lam=LAMBDA_BAL,
               acc_margin=ACC_MARGIN, n_exits=PLACEMENT_N, n_configurations=len(PLACEMENT_CONFIGS),
               depth_based=PLACEMENT_CONFIGS[0]["exits"], selected=MODEL_CFG["exit_layers"],
               depths=next(c["depths"] for c in PLACEMENT_CONFIGS if c["exits"] == MODEL_CFG["exit_layers"]))
    (OUTPUT_DIR / "exit_placement.json").write_text(json.dumps(rec, indent=1))
    print(f"Exit placement -> {OUTPUT_DIR / 'exit_placement.json'}")

## 8 · Attach and train exit heads

The backbone is frozen; only the exit heads and the final classifier tail are
trainable.

In [ ]:
def build_multi_exit_model(base_model, cfg):
    bb = next(l for l in base_model.layers if isinstance(l, tf.keras.Model))
    bb.trainable = False

    feats = [bb.get_layer(n).output for n in cfg["exit_layers"]]
    bb_multi = tf.keras.Model(bb.input, feats + [bb.output], name="bb_multi")

    inp = tf.keras.Input(shape=IMAGE_SIZE, name="image")
    x = tf.keras.layers.Rescaling(255.0, name="rescale")(inp) if cfg.get("expects_0_255") else inp
    outs = bb_multi(x)

    exits = [make_exit_head(outs[i], f"early_exit_{i+1}", exit_idx=i + 1)
             for i in range(len(feats))]

    tail = outs[-1]
    bb_idx = base_model.layers.index(bb)
    for layer in base_model.layers[bb_idx + 1:]:
        if FREEZE_FINAL_HEAD:
            layer.trainable = False
        tail = layer(tail)
    exits.append(tail)
    if not FREEZE_FINAL_HEAD:
        print("WARNING: final head is trainable.")
    return tf.keras.Model(inp, exits, name=f"{cfg['name']}_multi_exit")

multi_exit_model = build_multi_exit_model(base_model, MODEL_CFG)
n_out = len(multi_exit_model.outputs)
print("outputs:", multi_exit_model.output_names)

In [ ]:
import hashlib

model_dir = pathlib.Path(WEIGHTS_DIR).parent / OUTPUT_DIR.name
model_dir.mkdir(parents=True, exist_ok=True)

bb_fp = hashlib.sha256(pathlib.Path(weights_path).read_bytes()).hexdigest()[:8]
# specific to the exit layers, so that weights trained for other exits are never loaded
exit_tag = hashlib.sha1("|".join(MODEL_CFG["exit_layers"]).encode()).hexdigest()[:8]
ee_weights = model_dir / f"{RUN_TAG}_multi_exit_{exit_tag}.weights.h5"
print("backbone fingerprint:", bb_fp)

loss_weights = EE_LOSS_WEIGHTS[-n_out:]
multi_exit_model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss=["categorical_crossentropy"] * n_out,
    loss_weights=loss_weights,
    metrics=["accuracy"],
)
print("loss weights:", dict(zip(multi_exit_model.output_names, loss_weights)))

_probe = _ds(test_images, test_labels)

def check_final_matches_baseline(tag):
    if not FREEZE_FINAL_HEAD:
        return
    ev = dict(zip(multi_exit_model.metrics_names,
                  multi_exit_model.evaluate(_probe, verbose=0)))
    acc = ev[f"{multi_exit_model.output_names[-1]}_accuracy"]
    assert abs(acc - base_eval[1]) < 1e-3, (
        f"[{tag}] final exit {acc:.4f} != single-exit baseline {base_eval[1]:.4f}. "
        "The backbone/tail were overwritten (usually by a stale exit checkpoint) "
        "or the freeze did not take. Re-run the fine-tune cell and delete "
        f"{ee_weights}.")
    print(f"[{tag}] final-exit == baseline ({acc:.4f})")

if ee_weights.exists():
    multi_exit_model.load_weights(str(ee_weights))
    base_model.load_weights(weights_path)
    ee_history = None
    print(f"Loaded <- {ee_weights}")
else:
    ee_history = multi_exit_model.fit(train_ds, validation_data=val_ds,
                                      epochs=EPOCHS_EE, verbose=2)
    multi_exit_model.save_weights(str(ee_weights))
    print(f"Saved -> {ee_weights}")

check_final_matches_baseline("post-load")

plot_history(ee_history, f"{MODEL_CFG['name']} multi-exit - ",
             OUTPUT_DIR / f"{MODEL_CFG['name']}_multi_exit_history.png")

In [ ]:
ev = multi_exit_model.evaluate(_ds(test_images, test_labels), verbose=0)
names = multi_exit_model.output_names
accs  = ev[-n_out:]
for nm, a in zip(names, accs):
    print(f"{nm:<20s} test acc {a:.4f}")
print(f"{'single-exit baseline':<20s} test acc {base_eval[1]:.4f}")

## 9 · INT8 quantisation

In [ ]:
def representative_dataset():
    for batch, _ in full_ds.take(100):        # 100 batches of BATCH_SIZE images (12,800 with BATCH_SIZE = 128)
        yield [batch]

conv = tf.lite.TFLiteConverter.from_keras_model(multi_exit_model)
fp32_path = OUTPUT_DIR / f"{MODEL_CFG['name']}_multi_exit_fp32.tflite"
fp32_path.write_bytes(conv.convert())
print(f"fp32 -> {fp32_path} ({fp32_path.stat().st_size:,} B)")

int8_path = None
if QUANTIZE_FOR_TVM:
    conv = tf.lite.TFLiteConverter.from_keras_model(multi_exit_model)
    conv.optimizations             = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset    = representative_dataset
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type      = tf.float32
    conv.inference_output_type     = tf.float32
    int8_path = OUTPUT_DIR / f"{MODEL_CFG['name']}_multi_exit_int8.tflite"
    int8_path.write_bytes(conv.convert())
    print(f"int8 -> {int8_path} ({int8_path.stat().st_size:,} B)")

tflite_to_compile_path = int8_path if QUANTIZE_FOR_TVM else fp32_path
print("selected for TVM:", tflite_to_compile_path)

### FP32 vs. INT8 on the test set

Accuracy of every exit of the FP32 and INT8 models on the test images, their top-1 agreement and the mean change of
the confidence (maximum softmax probability) that the thresholds are compared with. TFLite orders the outputs
alphabetically, so they are matched by name, not by position. Then early exiting with the INT8 confidences for the
nine threshold profiles: accuracy, share of images at each exit and, for exits chosen by Algorithm 1, the mean cost
per image in its cost units. The best-EE profile is the fastest one within 2 points of the accuracy without exits.

In [ ]:
y_t = np.argmax(test_labels, axis=1)
res = {"FP32": DP.run_tflite(fp32_path, test_images)}
if int8_path is not None:
    res["INT8"] = DP.run_tflite(int8_path, test_images)
outs = multi_exit_model.output_names                 # early_exit_1 ... early_exit_N, then the final classifier
print(f"{'output':<16s}" + "".join(f"{k + ' acc':>11s}" for k in res)
      + ("   top-1 agree   mean |d conf|" if "INT8" in res else ""))
for n in outs:
    line = f"{n:<16s}" + "".join(f"{(r[n].argmax(1) == y_t).mean() * 100:10.2f}%" for r in res.values())
    if "INT8" in res:
        a, b = res["FP32"][n], res["INT8"][n]
        line += f"   {(a.argmax(1) == b.argmax(1)).mean() * 100:10.2f}%   {np.abs(a.max(1) - b.max(1)).mean():12.4f}"
    print(line)

if "INT8" in res and len(outs) > 1:
    q8 = res["INT8"]; ex = outs[:-1]
    cfg = next((c for c in PLACEMENT_CONFIGS if c["exits"] == MODEL_CFG["exit_layers"]), None)
    cost = [cum[i] + h for i, h in zip(cfg["idx"], cfg["head_costs"])] if cfg else [0.0] * len(ex)
    rows = DP.simulate([q8[n].max(1) for n in ex], [q8[n].argmax(1) for n in ex], q8[outs[-1]].argmax(1), y_t,
                       cost, cum[-1] if cfg else 0.0)
    print(f"\nEarly exiting, INT8 ({len(y_t)} test images):")
    for r in rows:
        print(f"  {r['profile']:<12s} acc {r['acc']:6.2f}%  exits {[round(s, 1) for s in r['exit_share']]}%"
              + (f"  cost {100 * r['cost'] / cum[-1]:5.1f}% of the backbone" if cfg else ""))
    if cfg:
        b = DP.best_ee(rows)
        print(f"Best-EE profile: {b['profile']} ({b['acc']:.2f}%, {100 * b['cost'] / cum[-1]:.1f}% of the backbone cost)")

## 10 · Compile to C

In [ ]:
import time

with open(tflite_to_compile_path, "rb") as f:
    tfl_bytes = f.read()
try:
    import tflite
    tfl_model = tflite.Model.GetRootAsModel(tfl_bytes, 0)
except AttributeError:
    import tflite.Model
    tfl_model = tflite.Model.Model.GetRootAsModel(tfl_bytes, 0)

mod, params = relay.frontend.from_tflite(tfl_model)
target   = tvm.target.Target("c")
runtime  = Runtime("crt")
executor = Executor("aot", {"unpacked-api": True, "interface-api": "c",
                            "workspace-byte-alignment": 8})
if params:
    mod["main"] = relay.build_module.bind_params_by_name(mod["main"], params)

USE_CUSTOM = MODEL_CFG.get("tvm_custom_pipeline", True)
OPT_LEVEL  = MODEL_CFG.get("tvm_opt_level", TVM_OPT_LEVEL)
t0 = time.time()

if USE_CUSTOM:
    print(f"custom fold-before-QNN pipeline, opt_level={OPT_LEVEL}")
    disabled = [] if QUANTIZE_FOR_TVM else ["qnn.Legalize", "qnn.Canonicalize"]
    with tvm.transform.PassContext(opt_level=OPT_LEVEL, disabled_pass=disabled,
                                   config={"tir.disable_vectorize": True}):
        mod = tvm.transform.Sequential([relay.transform.InferType(),
                                        relay.transform.FoldConstant(),
                                        relay.transform.SimplifyInference()])(mod)
        if QUANTIZE_FOR_TVM:
            mod = tvm.transform.Sequential([relay.qnn.transform.Legalize(),
                                            relay.qnn.transform.CanonicalizeOps()])(mod)
        compiled = relay.build(mod, target, runtime=runtime, executor=executor, params={})
else:
    print(f"standard pipeline, opt_level={OPT_LEVEL}")
    disabled = ["relay.AlterOpLayout"] + ([] if QUANTIZE_FOR_TVM
                                          else ["qnn.Legalize", "qnn.Canonicalize"])
    with tvm.transform.PassContext(opt_level=OPT_LEVEL, disabled_pass=disabled,
                                   config={"tir.disable_vectorize": True}):
        compiled = relay.build(mod, target, runtime=runtime, executor=executor, params={})

print(f"compiled in {time.time() - t0:.1f}s")
param_bytes = len(tvm.runtime.save_param_dict(params if params else compiled.get_params()))
print(f"params: {param_bytes:,} bytes")

tar = OUTPUT_DIR / (f"{MODEL_CFG['name']}_EE"
                    + ("_int8_aot.tar" if QUANTIZE_FOR_TVM else "_fp32_aot.tar"))
export_model_library_format(compiled, tar)
print(f"MLF -> {tar}")